# AI Skill Gap Classification using Decision Tree
This notebook follows the workflow from the provided project blueprint: Problem Definition → Dataset → Preprocessing → Feature Selection → Train/Test Split → Decision Tree → Evaluation → Prediction.

In [ ]:
# Import Libraries
import pandas as pd
import matplotlib.pyplot as plt
import joblib

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report


## Load Dataset

In [ ]:
df = pd.read_excel('Student_Skill_Gap_Dataset_100.xlsx')
print(df.head())
print("\nShape:", df.shape)


## Data Exploration

In [ ]:
print(df.info())
print(df.describe(include='all'))
print("\nMissing Values")
print(df.isnull().sum())
print("\nDuplicate Rows:", df.duplicated().sum())
df = df.drop_duplicates()


## Data Preprocessing

In [ ]:
le = LabelEncoder()
df['Internship'] = le.fit_transform(df['Internship'])
df['Industry_Readiness'] = le.fit_transform(df['Industry_Readiness'])
print(df.head())


## Feature Selection

In [ ]:
X = df[['CGPA','Python','Java','DSA','DBMS','OS','CN',
'Web_Development','Communication','Aptitude',
'Projects','Internship','Certifications']]

y = df['Industry_Readiness']


## Train Test Split

In [ ]:
X_train,X_test,y_train,y_test = train_test_split(
    X,y,test_size=0.25,random_state=42,stratify=y)
print("Training:",len(X_train))
print("Testing :",len(X_test))


## Train Decision Tree

In [ ]:
model = DecisionTreeClassifier(max_depth=4, random_state=42)
model.fit(X_train,y_train)


## Prediction and Evaluation

In [ ]:
y_pred = model.predict(X_test)

print("Accuracy:",accuracy_score(y_test,y_pred))

print("\nConfusion Matrix")
print(confusion_matrix(y_test,y_pred))

print("\nClassification Report")
print(classification_report(y_test,y_pred))


## Feature Importance

In [ ]:
importance = pd.DataFrame({
'Feature':X.columns,
'Importance':model.feature_importances_
}).sort_values('Importance',ascending=False)

print(importance)

plt.figure(figsize=(10,5))
plt.bar(importance['Feature'],importance['Importance'])
plt.xticks(rotation=45)
plt.title("Feature Importance")
plt.show()


## Decision Tree

In [ ]:
plt.figure(figsize=(20,10))
plot_tree(model,
          feature_names=X.columns,
          class_names=['Gap','Ready'],
          filled=True,
          rounded=True)
plt.show()


## Predict New Student

In [ ]:
new_student = pd.DataFrame({
'CGPA':[8.5],
'Python':[9],
'Java':[8],
'DSA':[8],
'DBMS':[8],
'OS':[7],
'CN':[7],
'Web_Development':[8],
'Communication':[9],
'Aptitude':[88],
'Projects':[3],
'Internship':[1],
'Certifications':[3]
})

prediction = model.predict(new_student)

if prediction[0]==1:
    print("Prediction: Ready")
else:
    print("Prediction: Gap")


## Save Predictions and Model

In [ ]:
df['Predicted_Status'] = model.predict(X)
df['Predicted_Status'] = df['Predicted_Status'].map({1:'Ready',0:'Gap'})

df.to_excel('Student_Predictions.xlsx',index=False)

joblib.dump(model,'skill_gap_model.pkl')

print("Prediction file saved: Student_Predictions.xlsx")
print("Model saved: skill_gap_model.pkl")


## Conclusion
The Decision Tree model predicts whether a student is Industry Ready or has a Skill Gap based on academic and technical skill attributes.